# 02 — Data Cleaning

**Goal:** turn the raw file into an analysis-ready dataset using the reproducible pipeline in
[`src/data_cleaning.py`](../src/data_cleaning.py), and verify every fix.

The logic lives in `src/`, not in this notebook, for one decisive reason: **the exact same code
must clean the training data, tomorrow's unseen data, and the rows a user types into the
Streamlit app.** Copy-pasted notebook cleaning is where train/serve skew comes from.

Every operation follows the same discipline:
1. what problem was found, 2. why it matters, 3. how it was handled, 4. how many records were affected.

**Nothing is dropped silently.** In fact, nothing is dropped at all — the cleaned dataset keeps
all 7,043 rows, and this notebook shows why that is the right call.


In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from src.config import CLEAN_DATA_FILE, TARGET_COLUMN
from src.data_cleaning import clean_telco, encode_target, load_raw

pd.set_option("display.max_columns", None)

## 1. Load and clean

`load_raw` validates the schema before anything else — if a future extract renames or drops a
column, we want a loud error, not a subtly wrong analysis.

In [2]:
raw = load_raw()
clean, steps = clean_telco(raw)

print(f"Raw:   {raw.shape[0]:,} rows x {raw.shape[1]} cols")
print(f"Clean: {clean.shape[0]:,} rows x {clean.shape[1]} cols")
print(f"Rows dropped: {raw.shape[0] - clean.shape[0]}\n")
for s in steps:
    print(s, end="\n\n")

Raw:   7,043 rows x 21 cols
Clean: 7,043 rows x 21 cols
Rows dropped: 0

[trim_whitespace] String cells may carry leading/trailing whitespace (in this file: only the blank TotalCharges values).
    action: str.strip() applied to every string column.
    rows affected: 11

[drop_duplicates] Exact duplicate rows would double-count customers.
    action: drop_duplicates() — none exist in the published dataset.
    rows affected: 0

[fix_total_charges] TotalCharges stored as text; 11 blank values hide from isna().
    action: Coerced to float; blanks (all tenure-0, verified) imputed as 0.0.
    rows affected: 11

[recode_senior_citizen] SeniorCitizen encoded 0/1 while sibling flags are No/Yes.
    action: Mapped {0: "No", 1: "Yes"}; now treated as categorical.
    rows affected: 7043

[validate_ranges] Corrupt extracts could carry impossible values.
    action: Checked tenure >= 0, MonthlyCharges > 0, TotalCharges >= 0, Churn in {Yes, No}. All passed.
    rows affected: 0



## 2. The decisions, one by one

### 2a. `TotalCharges`: coerce to numeric, impute the 11 blanks as 0 — not drop

- **Problem found (notebook 01):** stored as text; 11 rows contain a blank `" "`, invisible to `isna()`.
- **Why it matters:** a model cannot consume a string dollar amount, and hidden missingness would
  crash or corrupt downstream code.
- **How handled:** `pd.to_numeric(errors="coerce")`, then impute **0.0** — after first *verifying in
  code* that every blank row has `tenure = 0` (the pipeline raises an error otherwise).
- **Records affected: 11** (0.16% of the dataset).

**Why impute 0 instead of dropping the rows?** The choice was weighed, not defaulted:

| | Drop the 11 rows | Impute 0 (chosen) |
|---|---|---|
| Statistical effect | None measurable (0.16%) | None measurable |
| Semantic honesty | Treats structural blanks as bad data | 0 is the *true* amount billed so far |
| Segment coverage | Silently deletes every brand-new customer | Keeps the tenure-0 segment |
| Production readiness | Pipeline can't score new customers | Pipeline handles them naturally |

A deployed churn model will constantly meet customers with tenure 0. A cleaning rule that deletes
them would be a production bug, so we don't teach it here. Note this is *not* a general license to
impute missing data with 0 — it is justified only because the blank is structural: tenure = 0 ⟹
nothing billed yet ⟹ the true total is 0.

### 2b. `SeniorCitizen`: 0/1 → No/Yes

- **Problem:** the only demographic flag encoded 0/1 instead of Yes/No.
- **Why it matters:** consistency, and readable downstream output (plots, SHAP, app inputs).
  Mechanically the 0/1 version would work — this is a representation choice, not a correction.
- **How handled:** mapped `{0: "No", 1: "Yes"}`; now flows through the same categorical
  pipeline as `Partner`/`Dependents`.
- **Records affected: 7,043** (every row re-encoded; no information changed).

### 2c. Whitespace, duplicates, validation

- **Whitespace:** `str.strip()` on all string columns — affects only the 11 `TotalCharges` blanks
  in this file, but future-proofs the pipeline. (Notebook 01 verified no other column has
  whitespace or casing variants.)
- **Duplicates:** checked; **0 found**, so 0 dropped. The guard stays in the pipeline and *reports
  its count* rather than acting silently.
- **Validation:** `tenure ≥ 0`, `MonthlyCharges > 0`, `TotalCharges ≥ 0`, `Churn ∈ {Yes, No}` —
  all pass. On a corrupted future extract these raise errors instead of letting garbage flow in.

### 2d. What we deliberately do NOT do here

- **No target encoding in the saved file.** `Churn` stays Yes/No so the processed CSV is
  human-readable and directly usable for SQL and EDA. `encode_target()` (tested below) converts
  to 1/0 at modeling time.
- **No one-hot encoding, scaling, or feature engineering.** Those belong inside the sklearn
  `Pipeline` fit only on training folds (notebooks 05–06) — doing them here, on the full dataset,
  is exactly the pre-split leakage pattern we are avoiding.
- **No outlier removal.** Notebook 01 found no impossible values; \$118.75/month is a premium
  customer, not an error. Deleting inconvenient-but-real customers would bias the model.


## 3. Verify the fixes

Trust, but verify — each claim above is checked against the cleaned frame.

In [3]:
tc = clean["TotalCharges"]
assert tc.dtype == "float64" and tc.notna().all()
print(f"TotalCharges: dtype={tc.dtype}, NaN={tc.isna().sum()}, min={tc.min()}, max={tc.max():,.2f}")
print(f"Rows with TotalCharges == 0: {(tc == 0).sum()} (the 11 tenure-0 customers)")
assert (clean.loc[tc == 0, "tenure"] == 0).all()

print(f"\nSeniorCitizen values: {dict(clean['SeniorCitizen'].value_counts())}")
assert set(clean["SeniorCitizen"].unique()) == {"No", "Yes"}

assert clean.shape == raw.shape, "no rows or columns lost"
assert clean.duplicated().sum() == 0

y = encode_target(clean[TARGET_COLUMN])
print(f"\nencode_target: dtype={y.dtype}, positives={y.sum():,} ({y.mean():.2%}) — matches the 26.54% from notebook 01")

TotalCharges: dtype=float64, NaN=0, min=0.0, max=8,684.80
Rows with TotalCharges == 0: 11 (the 11 tenure-0 customers)

SeniorCitizen values: {'No': np.int64(5901), 'Yes': np.int64(1142)}

encode_target: dtype=int64, positives=1,869 (26.54%) — matches the 26.54% from notebook 01


## 4. Save the processed dataset

Saved as CSV (git-ignored). The file is also reproducible from scratch with one command:

```bash
python -m src.data_cleaning
```


In [4]:
CLEAN_DATA_FILE.parent.mkdir(parents=True, exist_ok=True)
clean.to_csv(CLEAN_DATA_FILE, index=False)
print(f"Saved {clean.shape[0]:,} rows -> {CLEAN_DATA_FILE.relative_to(CLEAN_DATA_FILE.parents[2])}")
clean.head()

Saved 7,043 rows -> data/processed/telco_churn_clean.csv


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,No,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,No,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,No,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,No,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 5. Summary

| Operation | Problem | Action | Rows affected |
|---|---|---|---|
| trim_whitespace | Blank strings hide missingness | strip all string cols | 11 |
| drop_duplicates | Would double-count customers | checked, none found | 0 |
| fix_total_charges | Text dtype + 11 hidden blanks | to float; blanks → 0.0 (verified tenure-0) | 11 |
| recode_senior_citizen | Inconsistent 0/1 encoding | map to No/Yes | 7,043 (re-encoded) |
| validate_ranges | Guard against corrupt extracts | invariant checks | 0 |

**7,043 rows in → 7,043 rows out.** The dataset is now typed correctly, free of hidden
missingness, consistently encoded, and validated — ready for EDA (notebook 03).
